# CHASM Usage Guide

In [ ]:
from pathlib import Path

# Define data directories
BASE_DIR = Path("./CHASM_data")
SWPC_SYNOPTIC_DRAWINGS_DIR = BASE_DIR / "swpc_synoptic_drawings"
SAM_SEGMENTATION_MASKS_DIR = BASE_DIR / "sam_segmentation_masks"
SDO_IMAGERY_DIR = BASE_DIR / "sdo_imagery"
CHASM_SELECTIONS_DIR = BASE_DIR / "chasm_selections"
SAM_CHECKPOINTS_DIR = BASE_DIR / "sam_checkpoints"

# Create directories if they don't exist
for directory in [
    SWPC_SYNOPTIC_DRAWINGS_DIR,
    SAM_SEGMENTATION_MASKS_DIR,
    SDO_IMAGERY_DIR,
    CHASM_SELECTIONS_DIR,
    SAM_CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

## SWPC Synoptic Drawings

In [ ]:
# Import the dataset class for the SWPC Synoptic Drawings
from chasm.data import DrawingsDataset

In [ ]:
# Get synoptic drawings from our Google Drive
drawings_dataset = DrawingsDataset(root=SWPC_SYNOPTIC_DRAWINGS_DIR, fetch_online=False)

In [ ]:
# Get synoptic drawings directly from the SWPC
from chasm.data.generate import SWPCDrawingsScraper

# Calling functions on this object saves the desired drawings to the save_path
drawings_scraper = SWPCDrawingsScraper(save_path=SWPC_SYNOPTIC_DRAWINGS_DIR)

# Get 1 day
# drawings_scraper.get_drawing_for_date("2026-01-01")

# Get a list of dates
# dates = ["2025-01-01", "2025-01-15", "2025-02-14"]
# drawings_scraper.get_drawings_for_dates(dates)

# Get a range of dates
start_date = "2025-01-01"
end_date = "2025-01-31"
drawings_scraper.get_drawings_for_date_range(start_date=start_date, end_date=end_date)

drawings_dataset = DrawingsDataset(root=SWPC_SYNOPTIC_DRAWINGS_DIR)

## SAM Masks

In [ ]:
# Import the dataset class for the SAM segmentation masks
from chasm.data import SAMMaskDataset

In [ ]:
# Get SAM segmentation masks from our Google Drive
masks_dataset = SAMMaskDataset(root=SAM_SEGMENTATION_MASKS_DIR, fetch_online=False)

In [ ]:
# Generate your own SAM segmentation masks
from chasm.data.generate import SAMSegmentationMasksGenerator

# Download a SAM checkpoint from https://github.com/facebookresearch/segment-anything?tab=readme-ov-file#model-checkpoints and put the file in the SAM_CHECKPOINTS_DIR
# vit_h is the largest, best version of SAM
# vit_b will work better on local systems

sam_mask_generator = SAMSegmentationMasksGenerator(
    swpc_drawing_dir=SWPC_SYNOPTIC_DRAWINGS_DIR,
    save_dir=SAM_SEGMENTATION_MASKS_DIR,
    model_type="vit_b",
    sam_checkpoint_filepath=SAM_CHECKPOINTS_DIR / "sam_vit_b_01ec64.pth",
)
# Create SAM segmentation masks for all drawings in the SWPC_SYNOPTIC_DRAWINGS_DIR and save them to SAM_SEGMENTATION_MASKS_DIR
sam_mask_generator.process_all_images()

masks_dataset = SAMMaskDataset(root=SAM_SEGMENTATION_MASKS_DIR)

## SDO Imagery

In [ ]:
# Import the dataset class for the SDO imagery
from chasm.data import SDODataset

In [ ]:
# Get SDO imagery from our Google Drive
sdo_dataset = SDODataset(root=SDO_IMAGERY_DIR / "full_size_images", fetch_online=False)

In [ ]:
# Download SDO imagery on your own
from chasm.data.generate import CHASMSDODownloader

FULL_SIZE_IMAGE_DIR = SDO_IMAGERY_DIR / "full_size_images"
POST_PROCESSED_IMAGE_DIR = SDO_IMAGERY_DIR / "post_processed_images"

# verbose=False (default): suppresses drms/downloader logs and shows tqdm progress bars
# verbose=True: shows full logging output instead
sdo_downloader = CHASMSDODownloader(
    full_save_path=FULL_SIZE_IMAGE_DIR,
    resampled_save_path=POST_PROCESSED_IMAGE_DIR,
    email="cbeckdevelopment@gmail.com",
    verbose=False,
)

swpc_dates = sdo_downloader.get_dates_from_swpc_drawing_dir(SWPC_SYNOPTIC_DRAWINGS_DIR)
jsoc_queries = sdo_downloader.get_jsoc_queries(swpc_dates)

download_results = sdo_downloader.download_images_parallel(jsoc_queries)

postprocess_results = sdo_downloader.post_process_parallel(download_results)

sdo_dataset = SDODataset(root=POST_PROCESSED_IMAGE_DIR)

## Run CHASM

In [ ]:
from chasm.gui.app import run_app

run_app(
    drawings_dataset=drawings_dataset,
    sam_dataset=masks_dataset,
    save_dir=CHASM_SELECTIONS_DIR,
)

### Or Download the CHASM Selections

In [ ]:
from chasm.data import CHASMDataset

chasm_selections = CHASMDataset(root=CHASM_SELECTIONS_DIR, fetch_online=True)

In [ ]:
# CHASM-1407

# CHASM-1111

# CHASM-967

## Post-Process CHASM Selections

## Train/Test CHRONNOS with CHASM Selections